In [0]:
from pyspark.sql import functions as F
from config.f1_config import CATALOG, SILVER_SCHEMA, GOLD_SCHEMA
GOLD_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.results"

In [0]:
df_silver_results = spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.results")
df_gold_dim_races = spark.read.table(f"{CATALOG}.{GOLD_SCHEMA}.races")
df_gold_dim_drivers = spark.read.table(f"{CATALOG}.{GOLD_SCHEMA}.drivers")
df_gold_dim_constructors = spark.read.table(f"{CATALOG}.{GOLD_SCHEMA}.constructors")


In [0]:
def build_fact_results(df_silver_results, df_silver_drivers, df_gold_dim_constructors, df_gold_dim_races):
    return(
        df_silver_results
        .join(df_gold_dim_drivers.select("Driver_Id"), on="Driver_Id", how="inner")
        .join(df_gold_dim_constructors.select("Constructor_Id"), on="Constructor_Id", how ="inner")
        .join(df_gold_dim_races.select("Season", "Round", "Circuit_Id"), on =["Season","Round"], how ="inner")
        .select("Driver_Id","Constructor_Id","Circuit_Id","Season","Round","Race_Name","Grid","Position",
            "Points","Laps", "Status", "Fastest_Lap_Rank")
    )

df_gold_fact_results = build_fact_results(df_silver_results, df_gold_dim_drivers, df_gold_dim_constructors, df_gold_dim_races)


In [0]:
df_gold_dim_races.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(GOLD_TABLE)